# H19b: Hessian-basis vs SV-basis partial equalization

## Notebook role

This notebook is the presentation and analysis companion to:

- `experiments/Tier_1_Core_Mechanism_Tests/H19b_HESSIAN_VS_SV_BASIS_ALIGNMENT/run_experiment.py`

It now **imports and uses the script's `run_experiment()` results directly** rather than re-implementing the numerical core. That makes the notebook and script a genuinely aligned pair.

## Truthful scope

This is a **tiny toy probe** on a 3-layer `4x4` deep linear network. It examines several update directions at a single warm-start point and measures:

- curvature-weighted alignment in the Hessian eigenbasis,
- how update energy is distributed across **algebraic Hessian eig-order sectors**, and
- a small outcome-linked metric: a **norm-matched one-step loss change** from the probe point.

It is **not** an end-to-end training study and **not** by itself a mechanistic proof of the broader Muon / SV-basis paradox.


## Reproducible setup and notebook-safe import path

This cell avoids `__file__` and explicitly locates the paired script in a notebook-safe way. It also logs the execution environment and then runs the shared experiment implementation.


In [ ]:
from pathlib import Path
import importlib.util
import platform
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
try:
    from IPython.display import Markdown, display
except ImportError:
    Markdown = lambda text: text
    def display(obj):
        print(obj)


TARGET_REL = Path("experiments/Tier_1_Core_Mechanism_Tests/H19b_HESSIAN_VS_SV_BASIS_ALIGNMENT/run_experiment.py")
EXPERIMENT_DIRNAME = "H19b_HESSIAN_VS_SV_BASIS_ALIGNMENT"


def locate_experiment_dir(start):
    start = start.resolve()
    for candidate in [start, *start.parents]:
        paired_script = candidate / TARGET_REL
        if paired_script.exists():
            return candidate, paired_script.parent
        if candidate.name == EXPERIMENT_DIRNAME and (candidate / "run_experiment.py").exists():
            experiment_dir = candidate
            repo_root = experiment_dir.parents[2]
            return repo_root, experiment_dir
    raise FileNotFoundError(
        "Could not locate the H19b experiment directory from the current working directory."
    )


repo_root, experiment_dir = locate_experiment_dir(Path.cwd())
script_path = experiment_dir / "run_experiment.py"

spec = importlib.util.spec_from_file_location("h19b_run_experiment", script_path)
h19b = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(h19b)

print(f"Repo root:       {repo_root}")
print(f"Experiment dir:  {experiment_dir}")
print(f"Script path:     {script_path}")
print(f"Python:          {sys.version.split()[0]}")
print(f"Platform:        {platform.platform()}")
print(f"NumPy:           {np.__version__}")
print(f"Pandas:          {pd.__version__}")

notebook_start = time.perf_counter()
results = h19b.run_experiment(verbose=False)
notebook_elapsed = time.perf_counter() - notebook_start

config = results["config"]
seed_df = pd.DataFrame(results["seed_rows"])
raw_df = pd.DataFrame(results["raw_rows"])
agg_df = pd.DataFrame(results["aggregate_rows"])
method_order = config["methods"]

print(f"Shared run_experiment() runtime: {results['runtime_seconds']:.3f} s")
print(f"Notebook cell wall time:         {notebook_elapsed:.3f} s")
print(f"Seeds:                          {config['seeds']}")
print(f"Methods:                        {', '.join(method_order)}")


## Protocol, methods, and caveats

The experiment keeps the original tiny setup almost unchanged:

- `3` linear layers, each `4x4`
- `48` total parameters
- `5` seeds
- warm-start with `50` momentum-SGD steps
- compute a full `48 x 48` Hessian by finite differences at the probe point
- compare six named update rules at that same point

Two caveats matter for interpretation:

1. The Hessian-sector metric is split by **algebraic eigenvalue order** from `np.linalg.eigh`, so the “lowest third” means the most negative/smallest eigenvalues, **not** necessarily the flattest directions by `|λ|`.
2. In this square `4x4` setting, `SV_full_k4` is expected to be exactly the same update as `Muon`, because both are the per-layer polar factor `U @ V^T`.


In [ ]:
config_df = pd.DataFrame([
    {"field": "dim", "value": config["dim"]},
    {"field": "n_layers", "value": config["n_layers"]},
    {"field": "n_params", "value": config["n_params"]},
    {"field": "warmup_steps", "value": config["warmup_steps"]},
    {"field": "num_seeds", "value": config["num_seeds"]},
    {"field": "learning_rate", "value": config["learning_rate"]},
    {"field": "momentum", "value": config["momentum"]},
    {"field": "fd_eps", "value": config["fd_eps"]},
    {"field": "one_step_loss_metric", "value": config["one_step_loss_metric"]},
])

method_df = pd.DataFrame(
    [{"method": m, "description": config["method_descriptions"][m]} for m in method_order]
)

notes_df = pd.DataFrame({"note": config["notes"]})
energy_sector_df = pd.DataFrame(
    [{"metric": k, "meaning": v} for k, v in config["energy_sector_labels"].items()]
)

display(Markdown("### Configuration"))
display(config_df)

display(Markdown("### Methods"))
display(method_df)

display(Markdown("### Energy-sector labels used in this notebook"))
display(energy_sector_df)

display(Markdown("### Notes / caveats returned by the script"))
display(notes_df)


## Per-seed diagnostics and reproducibility

These per-seed rows are the raw probe-point diagnostics returned by the script. They are useful for checking stability, Hessian sanity, and the `SV_full_k4 == Muon` identity claim.


In [ ]:
display(seed_df)

print("Prediction checks:")
for key, value in results["prediction_checks"].items():
    print(f"  {key}: {value}")

print("\nDiagnostics:")
for key, value in results["diagnostics"].items():
    print(f"  {key}: {value}")


## Aggregate metrics across methods

The table below summarizes the mean and standard deviation across seeds for the core measurements:

- **CWA** = curvature-weighted alignment using `|λ_i|`
- **Lowest / middle / highest eig third** = energy fractions in the Hessian eigenbasis split by **algebraic** eig-order
- **Norm-matched one-step Δloss** = `loss(θ - lr * normalized(update, ||g||)) - loss(θ)`

For the one-step loss metric, **more negative is better** because it means a larger loss decrease.


In [ ]:
agg_display = (
    agg_df.set_index("method")
    .loc[method_order, [
        "curvature_weighted_alignment_mean",
        "curvature_weighted_alignment_std",
        "lowest_eig_third_energy_mean",
        "lowest_eig_third_energy_std",
        "middle_eig_third_energy_mean",
        "middle_eig_third_energy_std",
        "highest_eig_third_energy_mean",
        "highest_eig_third_energy_std",
        "norm_matched_one_step_loss_delta_mean",
        "norm_matched_one_step_loss_delta_std",
    ]]
)

display(agg_display.round(6))


### Initial read of the aggregate table

A serious reading should focus on what the implementation **actually** measures:

- If `Hessian_partial_k10` does **not** beat `SV_partial_k2` on the returned sector metric, then the original notebook's story is not supported by this toy probe as currently implemented.
- If `SV_full_k4` and `Muon` match numerically, they should be interpreted as the same update in this specific square setting, not as two substantively different competitors.
- The one-step loss metric is only a small local utility check, but it is useful because it connects the geometry probe to an actual loss change.


In [ ]:
plot_df = agg_df.set_index("method").loc[method_order]

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(
    plot_df.index,
    plot_df["curvature_weighted_alignment_mean"],
    yerr=plot_df["curvature_weighted_alignment_std"],
    capsize=4,
    color="steelblue",
)
ax.set_ylabel("Curvature-weighted alignment")
ax.set_title("H19b: curvature-weighted alignment by method")
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


### Interpretation: curvature-weighted alignment

This plot is the cleanest single summary of how concentrated each update is in high-`|λ|` Hessian directions under the current CWA metric. It is still only a **local geometric descriptor**; by itself it does not establish causal optimization superiority.


In [ ]:
energy_cols = [
    "lowest_eig_third_energy_mean",
    "middle_eig_third_energy_mean",
    "highest_eig_third_energy_mean",
]
energy_plot_df = plot_df[energy_cols]

fig, ax = plt.subplots(figsize=(9, 5))
bottom = np.zeros(len(energy_plot_df))
colors = ["#d73027", "#fee08b", "#4575b4"]
labels = [
    "Lowest algebraic eig third",
    "Middle algebraic eig third",
    "Highest algebraic eig third",
]

for col, color, label in zip(energy_cols, colors, labels):
    ax.bar(energy_plot_df.index, energy_plot_df[col], bottom=bottom, label=label, color=color)
    bottom = bottom + energy_plot_df[col].to_numpy()

ax.set_ylabel("Mean energy fraction")
ax.set_title("H19b: Hessian-eigenbasis energy distribution by algebraic eig-order sector")
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
ax.set_ylim(0, 1.05)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, 1.18), ncol=1)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


### Interpretation: energy sectors

This figure should **not** be paraphrased as “flat vs sharp directions” unless the implementation is changed. In the current code, the sectors are just consecutive thirds of the Hessian eigenvectors in **algebraic** eigenvalue order:

- lowest third = most negative / smallest eigenvalues,
- middle third = central band,
- highest third = largest eigenvalues.

That makes this a still-useful but more limited signed-spectrum diagnostic.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(
    plot_df.index,
    plot_df["norm_matched_one_step_loss_delta_mean"],
    yerr=plot_df["norm_matched_one_step_loss_delta_std"],
    capsize=4,
    color="darkgreen",
)
ax.axhline(0.0, color="black", linewidth=1)
ax.set_ylabel("Norm-matched one-step Δloss")
ax.set_title("H19b: local one-step loss change from the warm-start point")
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

display(
    plot_df[["norm_matched_one_step_loss_delta_mean", "norm_matched_one_step_loss_delta_std"]]
    .round(8)
)


### Interpretation: one-step utility metric

This is the smallest outcome-linked addition in the first completion pass. Every method is norm-matched to `||g||` and then evaluated with the same step size from the same probe point. The metric is still local and toy-scale, but it provides a direct optimization quantity rather than only a basis-geometry surrogate.


## Method-equality diagnostic: `SV_full_k4` versus `Muon`

Because each layer is square `4x4`, both methods reduce to the same per-layer polar factor `U @ V^T`. The notebook should therefore treat them as **identical in this setup**, not as empirically distinct mechanisms.


In [ ]:
sv_full_rows = raw_df[raw_df["method"] == "SV_full_k4"].reset_index(drop=True)
muon_rows = raw_df[raw_df["method"] == "Muon"].reset_index(drop=True)
comparison_cols = [
    "curvature_weighted_alignment",
    "lowest_eig_third_energy",
    "middle_eig_third_energy",
    "highest_eig_third_energy",
    "norm_matched_one_step_loss_delta",
]

max_abs_diff = 0.0
for col in comparison_cols:
    max_abs_diff = max(max_abs_diff, float(np.max(np.abs(sv_full_rows[col] - muon_rows[col]))))

print("Returned script diagnostics:")
for key, value in results["diagnostics"].items():
    if "sv_full" in key:
        print(f"  {key}: {value}")

print(f"\nMax abs difference across selected per-seed metrics: {max_abs_diff:.3e}")


## Final conclusion

The cell below reports the conclusion from the **actual returned results** rather than from a pre-written narrative. This is important because the first completion pass is explicitly meant to stop overclaiming and remain honest if the current toy probe gives a negative or mixed result.


In [ ]:
checks = results["prediction_checks"]
agg_lookup = {row["method"]: row for row in results["aggregate_rows"]}

sv_partial_low = agg_lookup["SV_partial_k2"]["lowest_eig_third_energy_mean"]
sgd_low = agg_lookup["SGD"]["lowest_eig_third_energy_mean"]
hessian_partial_low = agg_lookup["Hessian_partial_k10"]["lowest_eig_third_energy_mean"]
sv_partial_cwa = agg_lookup["SV_partial_k2"]["curvature_weighted_alignment_mean"]
hessian_partial_cwa = agg_lookup["Hessian_partial_k10"]["curvature_weighted_alignment_mean"]

support_text = "supported" if checks["original_prediction_supported"] else "not supported"
verdict = "Current data support the original prediction." if checks["original_prediction_supported"] else "Current data do not support the original prediction under the present implementation."

conclusion_md = f"""
### Verdict

**{verdict}**

### What the current notebook can honestly claim

- This notebook now matches the paired script by importing and presenting the script's `run_experiment()` results directly.
- The probe is a **toy, warm-start, single-point geometric diagnostic** with a small local loss-change metric added for context.
- The Hessian-sector summaries are based on **algebraic eig-order thirds**, not absolute-curvature bins.
- `SV_full_k4` and `Muon` are numerically identical in this square `4x4` setting.

### What the current results say

- `SV_partial_k2` lowest-third energy mean: `{sv_partial_low:.3f}`
- `SGD` lowest-third energy mean: `{sgd_low:.3f}`
- `Hessian_partial_k10` lowest-third energy mean: `{hessian_partial_low:.3f}`
- `SV_partial_k2` CWA mean: `{sv_partial_cwa:.4f}`
- `Hessian_partial_k10` CWA mean: `{hessian_partial_cwa:.4f}`
- Original prediction status: **{support_text}**

### Calibrated interpretation

The first completion pass therefore upgrades the pair to a reproducible, aligned, and much more honest notebook/script package. It does **not** rescue the original strong mechanistic narrative. If a stronger claim is desired later, the next step should be a second pass that either changes the spectral metric to match the intended curvature semantics or expands the outcome-side evaluation beyond this toy local probe.
"""

display(Markdown(conclusion_md))
